In [1]:
import sys
import os
import asyncio
import numpy as np

sys.path.insert(0, os.path.abspath(".."))

from kinetic_core.engine.hpc_dispatcher import KineticHPCDispatcher

async def test_grid_sabotage():
    dispatcher = KineticHPCDispatcher()
    
    # Create 100x100 grid points
    temperatures = np.linspace(100, 2000, 100)
    pressures = np.linspace(0.01, 100.0, 100)
    
    # Measure time to just push to queue
    import time
    start = time.time()
    
    payloads, workers = await dispatcher.dispatch_tp_grid(temperatures, pressures)
    
    dispatch_time = time.time() - start
    
    print(f"Dispatched {len(payloads)} task futures.")
    print(f"Dispatch Time: {dispatch_time:.4f} seconds")
    
    assert len(payloads) == 10000, f"Expected 10,000 payloads, got {len(payloads)}"
    assert dispatch_time < 5.0, "Dispatch took too long! Inner/outer loop freezing Swarm node detected."
    
    print("SUCCESS: 10,000 grid nodes were mapped to async task futures instantly.")
    
    # Cleanup workers
    for w in workers:
        w.cancel()
        
await test_grid_sabotage()


Dispatched 10000 task futures.
Dispatch Time: 0.0063 seconds
SUCCESS: 10,000 grid nodes were mapped to async task futures instantly.
